In [ ]:
BASE_BERT_DIRECTORY = "../cache/baseline/bert"

In [ ]:
# Prepare Training and Test Dataset for MT-BERT
import pandas as pd
import os
from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd

def convert_to_mt_bert_df(df: pd.DataFrame):
    """
    Converts dataframe to MT BERT format
    :param df: Technical Debt formatted dataframe
    :return: Mt BERT formatted dataframe
    """
    df = df[["repository", "text", "label", "id"]] # to single line
    return df.rename(columns={"repository": "projectname", "text": "Abstract", "label": "original_label"})

def init_bert_data_directory(variant_name:str, training_type:str, dataset_name:str, train_df:pd.DataFrame, test_df:pd.DataFrame):
    """
    :param variant_name: Name of training variant
    :param training_type: Type of training variant(trained means technical debt training data pretrained means with other data)
    :param dataset_name: Unique or duplicate dataset name
    :param train_df: Training dataframe
    :param test_df: Testing dataframe
    """
    # Template for create 4 different types of training and test csv dataset files
    MT_BERT_FILE_FORMAT = BASE_BERT_DIRECTORY + "/input/satd/multi_train/{}_data/{}.csv"
    # All the relevant concatenated as prefix for later splitting and recovery
    # trained_bert_default_data
    # trained_unique_bert_5fcv1
    file_name_prefix = f"{training_type}_{dataset_name}_bert_{variant_name}"
    train_file = MT_BERT_FILE_FORMAT.format(file_name_prefix, f"{file_name_prefix}_code_comments_train")
    test_file = MT_BERT_FILE_FORMAT.format(file_name_prefix, f"{file_name_prefix}_code_comments_test")

    # Optionally save test dataset for post training testing
    post_train_test_file = f"{BASE_BERT_DIRECTORY}/input/unclassified_files/{file_name_prefix}_code_comments_test.csv"
    os.makedirs(os.path.dirname(post_train_test_file), exist_ok=True)
    convert_to_mt_bert_df(test_df).to_csv(post_train_test_file, index=False)


    # Create the real train and test file for code comment
    for file in [train_file, test_file]:
        os.makedirs(os.path.dirname(file), exist_ok=True)
    convert_to_mt_bert_df(train_df).to_csv(train_file, index=False)
    convert_to_mt_bert_df(test_df).to_csv(test_file, index=False)

    # Create dummy train and test files for the unused other three types as model input is mandatory
    for dataset_type in ["train", "test"]:
        for source in ["commit", "issue", "pr"]:
            other_file = MT_BERT_FILE_FORMAT.format(file_name_prefix, f"{file_name_prefix}_{source}_{dataset_type}")
            os.makedirs(os.path.dirname(other_file), exist_ok=True)
            convert_to_mt_bert_df(test_df)[:10].to_csv(other_file, index=False)

    # Command that will be executed for this training
    train_cmd = f"!{{sys.executable}} mt-bert-satd/run_mt-bert-satd-code-comment.py --data_dir {file_name_prefix} --output_dir model/{file_name_prefix}"
    # command that will be executed for thest testing
    test_cmd = f"!{{sys.executable}} mt-bert-satd/predict.py --task 4 --data_dir {file_name_prefix}_code_comments_test --output_dir output/{variant_name}/{training_type}/{dataset_name}/{dataset_name}"

    print(train_cmd)
    print(test_cmd)
    return file_name_prefix

# print("import sys")
# prepare training and test dataset for both unique and deuplicate dataset
for dataset_name in ["unique", "duplicate"]:
    full_train_df = pd.read_csv(f'../data/{dataset_name}_detect_train.csv')
    full_test_df = pd.read_csv(f'../data/{dataset_name}_detect_test.csv')
    # init training and testing dataset for default(without cross validation) experiment
    variant_directory = init_bert_data_directory("default", "trained", dataset_name, full_train_df, full_test_df)

    df = pd.concat([full_train_df, full_test_df])

    X = df["text"]
    y = df["label"]
    groups = df["repository"]

    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups)):
        train_df = df.iloc[train_idx]
        test_df = df.iloc[test_idx]
        fold_suffix = f"5fcv-{fold+1}"
        # print(f'Fold fold_suffix: {len(train_df)} train and {len(test_df)} test samples')
        # init training and testing dataset for 5 fold cross validation experiment
        variant_directory = init_bert_data_directory(fold_suffix, "trained", dataset_name, train_df, test_df)

    # Create test files for pretrained model
    post_train_test_file = f"{BASE_BERT_DIRECTORY}/input/unclassified_files/{dataset_name}_code_comments_test.csv"
    os.makedirs(os.path.dirname(post_train_test_file), exist_ok=True)
    convert_to_mt_bert_df(full_test_df).to_csv(post_train_test_file, index=False)





In [ ]:
import sys

py = sys.executable
folds = ["default", "5fcv-1", "5fcv-2", "5fcv-3", "5fcv-4", "5fcv-5"]
kinds = ["unique", "duplicate"]

for kind in kinds:
    for fold in folds:
        train_dir = f"trained_{kind}_bert_{fold}"
        test_dir  = f"{train_dir}_code_comments_test"

        !{py} mt-bert-satd/run_mt-bert-satd.py --data_dir {train_dir} --output_dir model/{train_dir}
        # No need to run prediction on newly trained model as after each iteration of the training the model runs test and we can simply take the last iteration's result.
        # !{py} mt-bert-satd/predict.py --task 4 --data_dir {test_dir} --output_dir output/{fold}/trained/{kind}/{kind}

In [ ]:
import sys
!{sys.executable} mt-bert-satd/predict.py --task 4 --data_dir unique_code_comments_test --output_dir predict_files/pretrained_unique_bert_default
!{sys.executable} mt-bert-satd/predict.py --task 4 --data_dir duplicate_code_comments_test --output_dir predict_files/pretrained_duplicate_bert_default


In [ ]:
from dotenv import load_dotenv
import os
from Model import Model
from constant import *
from SimpleOutputLabelConverter import SimpleOutputLabelConverter
load_dotenv()
total_detect_df = pd.concat([pd.read_csv(f'../data/duplicate_detect_train.csv'), pd.read_csv(f'../data/duplicate_detect_test.csv')])

def read_and_convert_output(bert_test_df: pd.DataFrame, train_type:str, dataset_name:str, model_name:str, flavour:str, bert_test_output_file:str):
    filtered = total_detect_df[total_detect_df["id"].isin(bert_test_df["id"])]
    # reorder
    result_df = filtered.set_index("id").loc[bert_test_df["id"]].reset_index()
    # check missing
    assert len(result_df) == len(bert_test_df), "Some IDs are missing"
    #Format: trained-liu-5fcv-4
    model = Model('detect', f"{train_type}-{model_name}-{flavour}", simple_output_label_converter, 10_000)
    # model.fit(detect_train_dataset)
    raw_predicted_labels = open(bert_test_output_file, 'r').readlines()
    raw_predicted_labels = list(map(lambda x: x.strip(), raw_predicted_labels))
    predicted_labels = list(map(lambda x: "yes" if x == "1" else "no", raw_predicted_labels))
    model.predict_end(Dataset.from_pandas(result_df), dataset_name, predicted_labels, raw_predicted_labels)

simple_output_label_converter = SimpleOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)



MODEL_DIRECTORY = f"{BASE_BERT_DIRECTORY}/model"


# Read and parse trained model output
print("Processing trained models output:")
for variant_dir in os.listdir(MODEL_DIRECTORY):
    parts = variant_dir.split("_")

    if len(parts) == 4 and not 'default' in variant_dir:
        print(variant_dir)
        train_type, dataset_name, model_name, flavour = parts
        bert_test_df_file = f"{BASE_BERT_DIRECTORY}/input/satd/multi_train/{variant_dir}_data/{variant_dir}_code_comments_test.csv"
        bert_test_df = pd.read_csv(bert_test_df_file)

        bert_test_output_file = f"{MODEL_DIRECTORY}/{variant_dir}/results_code_comments_8.txt"
        read_and_convert_output(bert_test_df, train_type, dataset_name, model_name, flavour, bert_test_output_file)


# print("Processing predicted output:")
# for variant_dir in os.listdir(f"{BASE_BERT_DIRECTORY}/predict_files"):
#     parts = variant_dir.split("_")
#
#     if len(parts) == 4:
#         print(variant_dir)
#         train_type, dataset_name, model_name, flavour = parts
#         dir_path = f"{BASE_BERT_DIRECTORY}/predict_files/{variant_dir}"
#         files = os.listdir(dir_path)
#
#         csv_files = [f for f in files if f.endswith(".csv")]
#         txt_files = [f for f in files if f.endswith(".txt")]
#
#         # strict assertions
#         assert len(csv_files) == 1, f"Expected 1 CSV, found {len(csv_files)}: {csv_files}"
#         assert len(txt_files) == 1, f"Expected 1 TXT, found {len(txt_files)}: {txt_files}"
#         assert len(files) == 2, f"Directory must contain exactly 2 files: {files}"
#
#         # full paths
#         csv_path = os.path.join(dir_path, csv_files[0])
#         txt_path = os.path.join(dir_path, txt_files[0])
#         read_and_convert_output(pd.read_csv(csv_path), train_type, dataset_name, model_name, flavour, txt_path)